<a href="https://colab.research.google.com/github/SamiraKheradmand/3-story-benchmark-transformer/blob/main/notebooks/02_experiments/window_random/transformer_highpass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import scipy.io as sio
from scipy.io.matlab._mio5_params import mat_struct
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from pathlib import Path
import re
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!ls "/content/drive/My Drive/ASCE_IASC"

figures					shm01a.mat  shm06a.mat
processed_zero_mean			shm02a.mat  shm07a.mat
random_block_assignment_45s_seed42.csv	shm03a.mat  shm08a.mat
random_window_split_seed42.csv		shm04a.mat  shm09a.mat
Results					shm05a.mat  txt_selected_sensors


In [ ]:
from pathlib import Path

import numpy as np
import scipy.io as sio


# مسیر فایل‌ها
DATA_DIR = Path("/content/drive/MyDrive/ASCE_IASC")

# فقط 15 سنسور واقعی سازه
CHANNELS = [f"DA{i:02d}" for i in range(1, 16)]

# پیدا کردن فایل‌های 9 حالت سازه
mat_files = sorted(DATA_DIR.glob("shm*.mat"))

print("Folder exists:", DATA_DIR.exists())
print("Number of files:", len(mat_files))
print("Number of channels:", len(CHANNELS))


def load_ambient_file(file_path):
    """
    خواندن یک فایل Ambient از داده‌های IASC-ASCE
    """

    mat = sio.loadmat(file_path,squeeze_me=True,struct_as_record=False)

    dasy = mat["dasy"]
    dasy_dscr = mat["dasy_dscr"]
    fs = float(mat["fsdasy"])

    signals = []
    descriptions = []

    for i, channel in enumerate(CHANNELS, start=1):

        signal = np.asarray(getattr(dasy, channel)).reshape(-1)

        description_name = f"DAdscr{i:02d}"
        description = getattr(dasy_dscr, description_name)

        signals.append(signal)
        descriptions.append(description)

    # شکل خروجی:
    # (تعداد نمونه‌های زمانی، تعداد سنسورها)
    X_raw = np.column_stack(signals)

    return X_raw, fs, descriptions

Folder exists: True
Number of files: 9
Number of channels: 15


In [ ]:
all_cases = {}

for file_path in mat_files:

    case_name = file_path.stem
    case_number = int(case_name[3:5])

    X_raw, fs, descriptions = load_ambient_file(file_path)

    all_cases[case_name] = {
        "X_raw": X_raw,
        "fs": fs,
        "channels": CHANNELS.copy(),
        "descriptions": descriptions,
        "case_number": case_number,
        "label": case_number - 1,
        "file_path": file_path
    }

    print(
        f"{case_name} | "
        f"shape: {X_raw.shape} | "
        f"fs: {fs} Hz | "
        f"label: {case_number - 1}"

    )
print( f"description: {pd.DataFrame(descriptions)}")

shm01a | shape: (60000, 15) | fs: 200.0 Hz | label: 0
shm02a | shape: (60000, 15) | fs: 200.0 Hz | label: 1
shm03a | shape: (60000, 15) | fs: 200.0 Hz | label: 2
shm04a | shape: (60000, 15) | fs: 200.0 Hz | label: 3
shm05a | shape: (60000, 15) | fs: 200.0 Hz | label: 4
shm06a | shape: (45568, 15) | fs: 200.0 Hz | label: 5
shm07a | shape: (180000, 15) | fs: 200.0 Hz | label: 6
shm08a | shape: (180000, 15) | fs: 200.0 Hz | label: 7
shm09a | shape: (180000, 15) | fs: 200.0 Hz | label: 8
description:                                                0
0   Base West side - EPI sensor X direction (N+)
1      Base Center - EPI sensor Y direction (W+)
2   Base East side - EPI sensor X direction (N+)
3    1st Floor - N/S EPI Sensor at West end (N+)
4      1st Floor - E/W FBA Sensor at Center (W+)
5    1st Floor - N/S FBA Sensor at East end (N+)
6    2nd Floor - N/S FBA Sensor at West end (N+)
7      2nd Floor - E/W EPI Sensor at Center (W+)
8    2nd Floor - N/S EPI Sensor at East end (N+)
9    3rd

In [ ]:
print(all_cases["shm01a"]["X_raw"].shape)
print(all_cases["shm01a"]["channels"])


(60000, 15)
['DA01', 'DA02', 'DA03', 'DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# ============================================================
# Step 2: Butterworth high-pass filter
# ============================================================

import pandas as pd
from scipy.signal import butter, sosfiltfilt


# ----------------------------
# Split settings
# ----------------------------
GAP_SEC = 0
EDGE_SEC = 2


# ----------------------------
# High-pass filter settings
# ----------------------------
CUTOFF = 0.1
ORDER = 2


def highpass_filter(X, fs):

    """
    Apply a Butterworth high-pass filter to all sensor channels.
    X shape:(samples, sensors)
    """
    sos = butter(N=ORDER, Wn=CUTOFF,btype="highpass",fs=fs,output="sos")
    X_filtered = sosfiltfilt(sos,X,axis=0)

    return X_filtered

In [ ]:
# ============================================================
# Step 3: Process complete signals
# ============================================================
USE_HIGHPASS = True
processed_data = {}


for case_name in sorted(all_cases.keys()):

    X_raw = all_cases[case_name]["X_raw"]
    fs = all_cases[case_name]["fs"]

    edge_size = int(EDGE_SEC * fs)

    # ----------------------------
    # Optional high-pass filter
    # ----------------------------
    if USE_HIGHPASS:
        X_processed = highpass_filter(X_raw,fs)

    else:
        X_processed = X_raw.copy()

    # ----------------------------
    # Matched edge removal
    # ----------------------------
    X_processed = X_processed[edge_size:-edge_size]
    processed_data[case_name] = X_processed


print("Signal processing completed.")

for case_name in sorted(processed_data.keys()):

    print(f"{case_name}: "
          f"{processed_data[case_name].shape}")

Signal processing completed.
shm01a: (59200, 15)
shm02a: (59200, 15)
shm03a: (59200, 15)
shm04a: (59200, 15)
shm05a: (59200, 15)
shm06a: (44768, 15)
shm07a: (179200, 15)
shm08a: (179200, 15)
shm09a: (179200, 15)


In [ ]:
# ============================================================
# Step 3: Select sensor channels
# ============================================================

# selected_sensor_indices = [8, 11, 13, 14]  # DA09, DA12, DA14, DA15
# selected_sensor_indices = [0,3]
selected_sensor_indices = [3,4,5,6,7,8,9,10,11,12,13,14]

selected_sensor_names = [CHANNELS[i] for i in selected_sensor_indices]

selected_data = {}

selected_data = {case_name: data[:, selected_sensor_indices]
                for case_name, data in processed_data.items()}


print("Selected sensors:", selected_sensor_names)

for case_name in sorted(selected_data.keys()):
    print(
        f"Case {case_name}: "
        f"data={selected_data[case_name].shape}, "
    )

Selected sensors: ['DA04', 'DA05', 'DA06', 'DA07', 'DA08', 'DA09', 'DA10', 'DA11', 'DA12', 'DA13', 'DA14', 'DA15']
Case shm01a: data=(59200, 12), 
Case shm02a: data=(59200, 12), 
Case shm03a: data=(59200, 12), 
Case shm04a: data=(59200, 12), 
Case shm05a: data=(59200, 12), 
Case shm06a: data=(44768, 12), 
Case shm07a: data=(179200, 12), 
Case shm08a: data=(179200, 12), 
Case shm09a: data=(179200, 12), 


In [ ]:
# ============================================================
# Step 5: Create all non-overlapping windows
# ============================================================

def make_all_windows(data_dict):
    """
    Create windows from all scenarios.

    Input:
        data_dict[case_name]
        shape = (samples, selected_sensors)

    Output:
        X shape = (windows, window_size, sensors)
        y shape = (windows,)
    """

    all_windows = []
    all_labels = []
    meta_rows = []

    global_window_id = 0

    for case_name in sorted(data_dict.keys()):

        X = data_dict[case_name]
        scenario_number = int(case_name[3:5])
        label = scenario_number - 1
        window_number = 0

        for start in range(0,len(X) - WINDOW_SIZE + 1,STEP_SIZE):

            end = start + WINDOW_SIZE
            window = X[start:end,:]

            all_windows.append(window)
            all_labels.append(label)

            meta_rows.append({
                "window_id": global_window_id,
                "case": case_name,
                "scenario": scenario_number,
                "label": label,
                "window": window_number,
                "start_sample": start,
                "end_sample": end })

            global_window_id += 1
            window_number += 1

    if len(all_windows) == 0:
        raise ValueError("No windows were created.")

    X_all = np.stack(all_windows,axis=0).astype(np.float32)
    y_all = np.array(all_labels,dtype=np.int64)
    meta_df = pd.DataFrame(meta_rows)

    return X_all, y_all, meta_df

In [ ]:
WINDOW_SIZE = 512
STEP_SIZE = 512

FS_DASY = 200
X_all, y_all, all_meta_df = make_all_windows(selected_data)

print("All windows shape:", X_all.shape)
print("All labels shape:", y_all.shape)

display(all_meta_df.head())

All windows shape: (1712, 512, 12)
All labels shape: (1712,)


,window_id,case,scenario,label,window,start_sample,end_sample
0,0,shm01a,1,0,0,0,512
1,1,shm01a,1,0,1,512,1024
2,2,shm01a,1,0,2,1024,1536
3,3,shm01a,1,0,3,1536,2048
4,4,shm01a,1,0,4,2048,2560


In [ ]:
# ============================================================
# Step 6: Random stratified window split
# ============================================================

all_indices = np.arange(len(X_all))

print("Total windows:", len(all_indices))


Total windows: 1712


In [ ]:
import random
import numpy as np
import torch

# ----------------------------
# Reproducibility
# ----------------------------
SEED = 2026

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducibility for CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Warn if a nondeterministic operation is used
    torch.use_deterministic_algorithms(True,warn_only=True)

set_seed(SEED)
print("Training seed:", SEED)

Training seed: 2026


In [ ]:
train_indices, temp_indices = train_test_split(
    all_indices,
    test_size=0.40,
    random_state=SEED,
    shuffle=True,
    stratify=y_all
)
stratify=y_all
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=y_all[temp_indices]
)

In [ ]:
X_train = X_all[train_indices]
y_train = y_all[train_indices]

X_val = X_all[val_indices]
y_val = y_all[val_indices]

X_test = X_all[test_indices]
y_test = y_all[test_indices]

In [ ]:
train_meta_df = (
    all_meta_df
    .iloc[train_indices]
    .reset_index(drop=True))

val_meta_df = (
    all_meta_df
    .iloc[val_indices]
    .reset_index(drop=True))

test_meta_df = (
    all_meta_df
    .iloc[test_indices]
    .reset_index(drop=True))

In [ ]:
print("Random Window Split completed.")

print("\nInput shapes:")
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nLabel shapes:")
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

Random Window Split completed.

Input shapes:
X_train: (1027, 512, 12)
X_val:   (342, 512, 12)
X_test:  (343, 512, 12)

Label shapes:
y_train: (1027,)
y_val:   (342,)
y_test:  (343,)


In [ ]:
total_windows = len(X_all)

print("Train ratio:",round(len(train_indices) / total_windows, 3))
print("Validation ratio:",round(len(val_indices) / total_windows, 3))
print("Test ratio:",round(len(test_indices) / total_windows, 3))

Train ratio: 0.6
Validation ratio: 0.2
Test ratio: 0.2


In [ ]:
def show_class_distribution(y, split_name):

    labels, counts = np.unique(y, return_counts=True)

    class_df = pd.DataFrame({
        "label": labels,
        "scenario": labels + 1,
        "windows": counts})

    print(f"\n{split_name} class distribution:")
    display(class_df)

In [ ]:
show_class_distribution(y_train,"Train")
show_class_distribution(y_val,"Validation")
show_class_distribution(y_test,"Test")


Train class distribution:


,label,scenario,windows
0,0,1,69
1,1,2,69
2,2,3,69
3,3,4,69
4,4,5,69
5,5,6,52
6,6,7,210
7,7,8,210
8,8,9,210



Validation class distribution:


,label,scenario,windows
0,0,1,23
1,1,2,23
2,2,3,23
3,3,4,23
4,4,5,23
5,5,6,17
6,6,7,70
7,7,8,70
8,8,9,70



Test class distribution:


,label,scenario,windows
0,0,1,23
1,1,2,23
2,2,3,23
3,3,4,23
4,4,5,23
5,5,6,18
6,6,7,70
7,7,8,70
8,8,9,70


In [ ]:
window_split_df = all_meta_df.copy()
window_split_df["split"] = ""

In [ ]:
window_split_df.loc[train_indices,"split"] = "train"
window_split_df.loc[val_indices,"split"] = "validation"
window_split_df.loc[test_indices,"split"] = "test"

In [ ]:
WINDOW_SPLIT_PATH = ("/content/drive/MyDrive/ASCE_IASC/"
    "random_window_split_seed42.csv")

window_split_df.to_csv(WINDOW_SPLIT_PATH,index=False)

print("Saved:", WINDOW_SPLIT_PATH)

Saved: /content/drive/MyDrive/ASCE_IASC/random_window_split_seed42.csv


In [ ]:
# ============================================================
# Step 4: Global train-based normalization
# One mean and standard deviation for each selected sensor
# ============================================================

EPS = 1e-8

print("Combined Train shape:", X_train.shape)
train_mean = X_train.mean(axis=(0,1),keepdims=True,dtype=np.float64)
train_std = X_train.std(axis=(0,1),keepdims=True,dtype=np.float64)


# Prevent division by zero
train_std_safe = np.where(train_std < EPS,1.0,train_std)


print("Train mean shape:", train_mean.shape)
print("Train std shape:", train_std_safe.shape)

Combined Train shape: (1027, 512, 12)
Train mean shape: (1, 1, 12)
Train std shape: (1, 1, 12)


In [ ]:
# ============================================================
# Step 4 (Continuation): Apply normalization to Train, Val, and Test
# ============================================================

# چون X_train، X_val و X_test آرایه‌های NumPy هستند،
# عملیات نرمال‌سازی به صورت برداری (Vectorized) و یکجا روی کل داده‌ها انجام می‌شود:

X_train_norm = ((X_train - train_mean) / train_std_safe).astype(np.float32)
X_val_norm   = ((X_val   - train_mean) / train_std_safe).astype(np.float32)
X_test_norm  = ((X_test  - train_mean) / train_std_safe).astype(np.float32)

print("Normalization completed.")

# ------------------------------------------------------------
# Check normalization results on Train set
# ------------------------------------------------------------
normalized_train_mean = X_train_norm.mean(axis=(0, 1)) # میانگین باید خیلی نزدیک به صفر شود
normalized_train_std  = X_train_norm.std(axis=(0, 1))  # انحراف معیار باید خیلی نزدیک به یک شود

print("Train mean after normalization:")
print(np.round(normalized_train_mean, 5))

print("\nTrain std after normalization:")
print(np.round(normalized_train_std, 5))

Normalization completed.
Train mean after normalization:
[ 0. -0. -0. -0. -0. -0.  0. -0. -0.  0. -0.  0.]

Train std after normalization:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
# ============================================================
# Step 6: Convert arrays to PyTorch tensors
# ============================================================

import random
import numpy as np
import torch

from torch.utils.data import TensorDataset, DataLoader



# Inputs must be float32
X_train_tensor = torch.from_numpy(X_train_norm.astype(np.float32, copy=False))
X_val_tensor = torch.from_numpy(X_val_norm.astype(np.float32, copy=False))
X_test_tensor = torch.from_numpy(X_test_norm.astype(np.float32, copy=False))


# Labels for CrossEntropyLoss must be int64
y_train_tensor = torch.from_numpy(y_train.astype(np.int64, copy=False))
y_val_tensor = torch.from_numpy(y_val.astype(np.int64, copy=False))
y_test_tensor = torch.from_numpy(y_test.astype(np.int64, copy=False))


print("\nTensor shapes:")
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

print("X_val_tensor:", X_val_tensor.shape)
print("y_val_tensor:", y_val_tensor.shape)

print("X_test_tensor:", X_test_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)


Tensor shapes:
X_train_tensor: torch.Size([1027, 512, 12])
y_train_tensor: torch.Size([1027])
X_val_tensor: torch.Size([342, 512, 12])
y_val_tensor: torch.Size([342])
X_test_tensor: torch.Size([343, 512, 12])
y_test_tensor: torch.Size([343])


In [ ]:
# ============================================================
# Create Dataset and DataLoader
# ============================================================

train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,y_test_tensor)

BATCH_SIZE = 64

# برای تکرارپذیری Shuffle داده‌های Train
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)


train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,generator=loader_generator)
val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader  = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

In [ ]:
print("Number of train samples:", len(train_dataset))
print("Number of val samples:  ", len(val_dataset))
print("Number of test samples: ", len(test_dataset))

print("\nNumber of train batches:", len(train_loader))
print("Number of val batches:  ", len(val_loader))
print("Number of test batches: ", len(test_loader))


X_batch, y_batch = next(iter(train_loader))

print("\nOne train batch:")
print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First 10 labels:", y_batch[:10])

Number of train samples: 1027
Number of val samples:   342
Number of test samples:  343

Number of train batches: 17
Number of val batches:   6
Number of test batches:  6

One train batch:
X_batch shape: torch.Size([64, 512, 12])
y_batch shape: torch.Size([64])
First 10 labels: tensor([6, 4, 8, 6, 0, 6, 1, 1, 1, 7])


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

# ----------------------------
# Device
# ----------------------------
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)



Device: cuda


In [ ]:
# ============================================================
# Step 7: Define raw time-series Transformer
# Input shape: (batch, time, sensors)
# ============================================================

import torch.nn as nn

# --- Added these definitions to ensure the model parameters are defined ---
NUM_CLASSES = 9
SEQ_LEN = WINDOW_SIZE
INPUT_DIM = X_train_tensor.shape[2]
D_MODEL = 64
NUM_HEADS = 8
NUM_LAYERS = 2
DIM_FEEDFORWARD = 64
DROPOUT = 0.10
# --- End of added definitions ---

class TransformerTimeSeriesClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        seq_len,
        num_classes,
        d_model=64,
        num_heads=4,
        num_layers=4,
        dim_feedforward=128,
        dropout=0.10
    ):
        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model

        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model,num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embedding, mean=0.0, std=0.02)

    def forward(self, x):
        # x shape: (batch, time, channels)

        x = self.input_projection(x)
        x = x + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer_encoder(x)
        x = self.final_norm(x)
        # Mean pooling over time
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits

set_seed(SEED)
model = TransformerTimeSeriesClassifier(
    input_dim=INPUT_DIM,
    seq_len=SEQ_LEN,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(device)

print(model)

# ----------------------------
# Count trainable parameters
# ----------------------------
def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nTrainable parameters:", count_trainable_parameters(model))


# ----------------------------
# Check one forward pass
# ----------------------------
for xb, yb in train_loader:
    xb = xb.to(device)

    logits = model(xb)

    print("\nInput batch shape:", xb.shape)
    print("Output logits shape:", logits.shape)

    break

TransformerTimeSeriesClassifier(
  (input_projection): Linear(in_features=12, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=64, out_features=9, bias=True)
)

Trainable parameters: 84745

Input batch shape:

/tmp/ipykernel_635/2861531204.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)


In [ ]:
# ============================================================
# Step 8: Loss function and optimizer
# ============================================================

class_counts = torch.bincount(y_train_tensor, minlength=NUM_CLASSES).float()
if torch.any(class_counts == 0):
    raise ValueError("At least one class has no training samples.")

class_weights = (len(y_train_tensor) / (NUM_CLASSES * class_counts))
class_weights = class_weights.to(device)

print("Class counts and weights:")

for class_id in range(NUM_CLASSES):

    print(
        f"Scenario {class_id + 1} | "
        f"count: {int(class_counts[class_id])} | "
        f"weight: {class_weights[class_id].item():.4f}"
    )

Class counts and weights:
Scenario 1 | count: 69 | weight: 1.6538
Scenario 2 | count: 69 | weight: 1.6538
Scenario 3 | count: 69 | weight: 1.6538
Scenario 4 | count: 69 | weight: 1.6538
Scenario 5 | count: 69 | weight: 1.6538
Scenario 6 | count: 52 | weight: 2.1944
Scenario 7 | count: 210 | weight: 0.5434
Scenario 8 | count: 210 | weight: 0.5434
Scenario 9 | count: 210 | weight: 0.5434


In [ ]:
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Loss function: CrossEntropyLoss()
Optimizer: AdamW
Learning rate: 0.0002


In [ ]:
# ============================================================
# Step 9: Train Transformer
# Best model selection based on Validation Macro-F1
# ============================================================

from sklearn.metrics import accuracy_score, f1_score
import copy
import torch

NUM_EPOCHS = 100
PATIENCE = 20
GRAD_CLIP = 1.0
MIN_DELTA = 1e-4

LABELS_ORDER = np.arange(NUM_CLASSES)

transformer_history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "train_macro_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": [],
}

best_val_macro_f1 = -1.0
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0


def run_one_epoch_transformer(model, data_loader, criterion, optimizer=None, device="cpu"):
    """
    If optimizer is given: training mode
    If optimizer is None: evaluation mode
    """

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for xb, yb in data_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(xb)
            loss = criterion(logits, yb)

            if is_training:
                loss.backward()

                # Gradient clipping helps Transformer training stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

                optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(yb.detach().cpu())

    avg_loss = total_loss / len(data_loader.dataset)
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return avg_loss, acc, macro_f1


for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc, train_macro_f1 = run_one_epoch_transformer(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_acc, val_macro_f1 = run_one_epoch_transformer(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        optimizer=None,
        device=device
    )

    transformer_history["epoch"].append(epoch)
    transformer_history["train_loss"].append(train_loss)
    transformer_history["train_acc"].append(train_acc)
    transformer_history["train_macro_f1"].append(train_macro_f1)
    transformer_history["val_loss"].append(val_loss)
    transformer_history["val_acc"].append(val_acc)
    transformer_history["val_macro_f1"].append(val_macro_f1)

    # Save best model based on validation Macro-F1
    if val_macro_f1 > best_val_macro_f1 + MIN_DELTA:
        best_val_macro_f1 = val_macro_f1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Train Macro-F1: {train_macro_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f}"
        )

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

assert best_model_state is not None, "No best model was saved."

# Load best validation model
model.load_state_dict(best_model_state)

print("\nTraining finished.")
print("Best validation Macro-F1:", best_val_macro_f1)

In [ ]:
if best_model_state is None:
    raise RuntimeError("No best model was saved.")

model.load_state_dict(best_model_state)

print("\nTraining finished.")
print("Best epoch:",best_epoch)
print("Best validation Macro-F1:",best_val_macro_f1)

In [ ]:
# ============================================================
# Step 10: Final evaluation
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import torch


train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

def predict_with_model_transformer(model, data_loader, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    return all_labels, all_preds


def final_evaluate_transformer(model, data_loader, split_name, device="cpu"):
    y_true, y_pred = predict_with_model_transformer(
        model=model,
        data_loader=data_loader,
        device=device
    )

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n===== {split_name} =====")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")

    labels_order = np.arange(NUM_CLASSES)
    target_names = [f"Scenario {i+1}" for i in labels_order]

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=target_names,
            labels=labels_order,
            digits=4,
            zero_division=0
        )
    )

    return {
        "split": split_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }


model.eval()

transformer_train_results = final_evaluate_transformer(
    model, train_eval_loader, "TRAIN", device=device
)

transformer_val_results = final_evaluate_transformer(
    model, val_loader, "VALIDATION", device=device
)

transformer_test_results = final_evaluate_transformer(
    model, test_loader, "TEST", device=device
)

transformer_summary_results = pd.DataFrame([
    {
        "model": "Transformer",
        "split": transformer_train_results["split"],
        "accuracy": transformer_train_results["accuracy"],
        "macro_f1": transformer_train_results["macro_f1"],
        "weighted_f1": transformer_train_results["weighted_f1"],
    },
    {
        "model": "Transformer",
        "split": transformer_val_results["split"],
        "accuracy": transformer_val_results["accuracy"],
        "macro_f1": transformer_val_results["macro_f1"],
        "weighted_f1": transformer_val_results["weighted_f1"],
    },
    {
        "model": "Transformer",
        "split": transformer_test_results["split"],
        "accuracy": transformer_test_results["accuracy"],
        "macro_f1": transformer_test_results["macro_f1"],
        "weighted_f1": transformer_test_results["weighted_f1"],
    }
])

display(transformer_summary_results)

In [ ]:
transformer_summary_results = pd.DataFrame([
    {
        "model":"Raw Transformer",
        "preprocessing":
            "Butterworth high-pass + "
            "Train-based normalization",
        "selected_sensors": len(selected_sensor_names),
        "split": transformer_train_results["split"],
        "accuracy":transformer_train_results["accuracy"],
        "macro_f1": transformer_train_results["macro_f1"],
        "weighted_f1": transformer_train_results["weighted_f1"]
    },

    {
        "model":"raw Transformer",
        "preprocessing":
            "Butterworth high-pass + "
            "Train-based normalization",

        "selected_sensors":len(selected_sensor_names),
        "split":transformer_val_results["split"],
        "accuracy":transformer_val_results["accuracy"],
        "macro_f1":transformer_val_results["macro_f1"],
        "weighted_f1":transformer_val_results["weighted_f1"]
    },

    {
        "model":
            "Raw Transformer",
        "preprocessing":
            "Butterworth high-pass + "
            "Train-based normalization",
        "selected_sensors":len(selected_sensor_names),
        "split":transformer_test_results["split"],
        "accuracy":transformer_test_results["accuracy"],
        "macro_f1":transformer_test_results["macro_f1"],
        "weighted_f1":transformer_test_results["weighted_f1"]
    }
])


display(transformer_summary_results)

In [ ]:
# ============================================================
# Step 11: Test confusion matrix
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# True and predicted labels from Transformer test results
y_true_test = transformer_test_results["y_true"]
y_pred_test = transformer_test_results["y_pred"]

labels_order = np.arange(NUM_CLASSES)
display_labels = [f"S{i+1}" for i in labels_order]

cm_transformer_test = confusion_matrix(
    y_true_test,
    y_pred_test,
    labels=labels_order
)

plt.figure(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_transformer_test,
    display_labels=display_labels
)

disp.plot(
    values_format="d",
    cmap="Blues",
    colorbar=False
)

plt.title("Test Confusion Matrix\n"
    "Transformer with Butterworth High-pass Filtering")
plt.xlabel("Predicted Scenario")
plt.ylabel("True Scenario")
plt.grid(False)
plt.show()

print("Transformer test confusion matrix:")
print(cm_transformer_test)

In [ ]:
# ============================================================
# Step 12: Training curves
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    transformer_history["epoch"],
    transformer_history["train_loss"],
    label="Train Loss"
)

plt.plot(
    transformer_history["epoch"],
    transformer_history["val_loss"],
    label="Validation Loss"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    transformer_history["epoch"],
    transformer_history["train_macro_f1"],
    label="Train Macro-F1"
)

plt.plot(
    transformer_history["epoch"],
    transformer_history["val_macro_f1"],
    label="Validation Macro-F1"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Training and Validation Macro-F1")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()